# Phase 2 – Data Quality, Feature Engineering, EDA & Hypothesis Testing
**Sigmoid case study – Bike Sharing Demand**

**Business question:** *Should the company buy more bikes, or can availability be improved by understanding and managing demand?*

This notebook:
1. applies the cleaning decisions found in Phase 1,
2. engineers features (each with a business reason),
3. explores demand with 12 charts, each with an interpretation,
4. states and tests one hypothesis.

Run with `Runtime → Run all`. The first cell asks for the dataset ZIP.

## 0. Setup – upload data and create the project modules

In [ ]:
import os, glob, zipfile, sys

if not (os.path.exists("data/hour.csv") and os.path.exists("data/day.csv")):
    try:
        from google.colab import files
        uploaded = files.upload()                       # choose the bike-sharing ZIP
        zip_name = [f for f in uploaded if f.lower().endswith(".zip")][0]
    except ImportError:
        zip_name = glob.glob("*.zip")[0]
    os.makedirs("data", exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        for member in z.namelist():
            name = os.path.basename(member)
            if name in ("hour.csv", "day.csv", "Readme.txt"):
                with z.open(member) as src, open(os.path.join("data", name), "wb") as dst:
                    dst.write(src.read())

for d in ["src", "outputs/charts", "outputs/model_results"]:
    os.makedirs(d, exist_ok=True)
print(sorted(os.listdir("data")))

The next three cells write the reusable project modules to `src/` (the same files delivered in the project folder), so this notebook is self-contained in Colab.

In [ ]:
%%writefile src/preprocessing.py
"""
Loading and cleaning for the Bike Sharing dataset.

Rules followed:
* Raw CSVs are never modified on disk; every function returns a new DataFrame.
* Nothing is deleted. Bad values are corrected and FLAGGED so the change is visible.
* Every cleaning decision is listed in CLEANING_LOG (shown in the notebook / report).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

# Readme: values were divided by these maxima -> multiply back for human-readable units
TEMP_MAX_C, ATEMP_MAX_C, HUM_MAX, WIND_MAX = 41, 50, 100, 67

# Days on which the system was (partly) shut down or data are incomplete (< 22 recorded hours).
# Identified in Phase 1; kept for modelling but flagged.
DISRUPTED_DAYS = pd.to_datetime([
    "2011-01-18", "2011-01-26", "2011-01-27", "2011-02-22",
    "2011-08-27", "2011-08-28", "2012-10-29", "2012-10-30",
])

CLEANING_LOG = [
    ("hum == 0 on 2011-03-10 (22 hourly rows, 1 daily row)",
     "Physically impossible -> sensor error. Replaced with the mean humidity of the same hour on "
     "the previous and next day (daily file: mean of neighbouring days). Flag column `hum_imputed`."),
    ("weathersit == 4 (3 hourly rows)",
     "Too rare to learn from. New column `weather` merges 4 into 3 ('rain/snow'). Raw column kept."),
    ("windspeed == 0 (12.5% of hours; next value is 0.0896)",
     "Likely 'calm / not recorded'. Kept unchanged, flagged in `wind_zero`. Low impact on demand."),
    ("165 missing hours (mostly 02:00-05:00; storm shutdowns)",
     "Not filled with invented rows for EDA/model training. Zero-rental hours appear to be absent, "
     "so night-time averages are slightly overstated. Storm days flagged in `disrupted_day`."),
    ("High cnt values (up to 977/hr)",
     "Genuine commute peaks (Phase 1, Step 10). Not removed."),
]


def load_hour(path: str = "data/hour.csv") -> pd.DataFrame:
    return pd.read_csv(path, parse_dates=["dteday"])


def load_day(path: str = "data/day.csv") -> pd.DataFrame:
    return pd.read_csv(path, parse_dates=["dteday"])


def _impute_hourly_humidity(df: pd.DataFrame) -> pd.DataFrame:
    bad = df["hum"] == 0
    df["hum_imputed"] = bad.astype(int)
    if not bad.any():
        return df
    lookup = df.loc[~bad].set_index(["dteday", "hr"])["hum"]
    for idx in df.index[bad]:
        d, h = df.at[idx, "dteday"], df.at[idx, "hr"]
        neighbours = [lookup.get((d + pd.Timedelta(days=k), h)) for k in (-1, 1)]
        neighbours = [v for v in neighbours if v is not None and not pd.isna(v)]
        df.at[idx, "hum"] = np.mean(neighbours) if neighbours else df.loc[~bad, "hum"].median()
    return df


def clean_hour(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy().sort_values(["dteday", "hr"]).reset_index(drop=True)
    df["timestamp"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
    df = _impute_hourly_humidity(df)
    df["weather"] = df["weathersit"].clip(upper=3)
    df["wind_zero"] = (df["windspeed"] == 0).astype(int)
    df["disrupted_day"] = df["dteday"].isin(DISRUPTED_DAYS).astype(int)
    return add_physical_units(df)


def clean_day(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy().sort_values("dteday").reset_index(drop=True)
    bad = df["hum"] == 0
    df["hum_imputed"] = bad.astype(int)
    for idx in df.index[bad]:
        neigh = df.loc[[i for i in (idx - 1, idx + 1) if i in df.index and not bad[i]], "hum"]
        df.at[idx, "hum"] = neigh.mean()
    df["weather"] = df["weathersit"].clip(upper=3)
    df["disrupted_day"] = df["dteday"].isin(DISRUPTED_DAYS).astype(int)
    return add_physical_units(df)


def add_physical_units(df: pd.DataFrame) -> pd.DataFrame:
    """De-normalise for readable charts and for the prototype (no new information)."""
    df["temp_c"] = df["temp"] * TEMP_MAX_C
    df["atemp_c"] = df["atemp"] * ATEMP_MAX_C
    df["hum_pct"] = df["hum"] * HUM_MAX
    df["wind_unscaled"] = df["windspeed"] * WIND_MAX   # Readme does not state the unit
    return df


In [ ]:
%%writefile src/feature_engineering.py
"""
Feature engineering. Every feature uses only information known BEFORE the hour starts
(calendar + weather for that hour, which in deployment comes from a weather forecast).
No feature uses `casual`, `registered`, or future rows.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

MORNING_PEAK = (7, 8, 9)
EVENING_PEAK = (16, 17, 18, 19)
PEAK_HOURS = MORNING_PEAK + EVENING_PEAK
NIGHT_HOURS = (0, 1, 2, 3, 4, 5)

FEATURE_RATIONALE = {
    "hr":            "Hour of day - the single strongest driver (commute vs night).",
    "weekday":       "Day of week (0=Sun). Captures Fri/Mon effects beyond the working-day flag.",
    "workingday":    "Commuting happens on working days; changes the whole daily shape.",
    "holiday":       "Public holidays behave like weekends even on Mon-Fri.",
    "is_weekend":    "Sat/Sun indicator - leisure pattern (midday peak).",
    "mnth":          "Month - seasonal temperature/daylight cycle.",
    "season":        "Coarser seasonal signal from the source data.",
    "yr":            "Growth of the system (2012 ~ +63%). Lets the model learn the level shift.",
    "morning_peak":  "Working day AND 07-09 - morning commute window.",
    "evening_peak":  "Working day AND 16-19 - evening commute window.",
    "rush_hour":     "Either commute window on a working day - where shortages are most likely.",
    "night":         "00-05 - near-zero demand; useful window for rebalancing trucks.",
    "hr_sin/hr_cos": "Cyclical hour: tells linear models that 23:00 is next to 00:00.",
    "mnth_sin/mnth_cos": "Cyclical month: December is next to January.",
    "weather":       "1 clear, 2 mist/cloud, 3 rain/snow (4 merged into 3 - only 3 rows).",
    "temp":          "Temperature (normalised). Riding is weather-sensitive.",
    "temp_sq":       "Lets a linear model capture 'too hot' - demand flattens/falls at high temps.",
    "hum":           "Humidity - proxy for discomfort and rain.",
    "windspeed":     "Wind - discomfort on a bike.",
}

EXCLUDED_COLUMNS = {
    "casual":      "Leakage: casual + registered = cnt exactly. Unknown before the hour ends.",
    "registered":  "Leakage: same reason as casual.",
    "cnt":         "Target.",
    "instant":     "Row index; it is a disguised time trend with no causal meaning. `yr` covers the trend explicitly.",
    "dteday":      "Raw date string; decomposed into yr/mnth/weekday instead.",
    "timestamp":   "Helper for sorting/splitting, not a feature.",
    "atemp":       "'Feels-like' temperature, correlation with temp ~0.99 -> redundant; kept out to avoid collinearity.",
    "weathersit":  "Replaced by `weather` (category 4 merged into 3).",
    "hum_imputed / wind_zero / disrupted_day": "Data-quality flags for analysis, not demand drivers.",
    "temp_c / atemp_c / hum_pct / wind_unscaled": "Same information as the scaled columns, used only for readable charts/app.",
}

MODEL_FEATURES = [
    "hr", "weekday", "workingday", "holiday", "is_weekend", "mnth", "season", "yr",
    "morning_peak", "evening_peak", "rush_hour", "night",
    "hr_sin", "hr_cos", "mnth_sin", "mnth_cos",
    "weather", "temp", "temp_sq", "hum", "windspeed",
]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["is_weekend"] = out["weekday"].isin([0, 6]).astype(int)
    if "hr" in out:
        hr = out["hr"]
        out["morning_peak"] = (hr.isin(MORNING_PEAK) & (out["workingday"] == 1)).astype(int)
        out["evening_peak"] = (hr.isin(EVENING_PEAK) & (out["workingday"] == 1)).astype(int)
        out["rush_hour"] = (out["morning_peak"] | out["evening_peak"]).astype(int)
        out["night"] = hr.isin(NIGHT_HOURS).astype(int)
        out["hr_sin"] = np.sin(2 * np.pi * hr / 24)
        out["hr_cos"] = np.cos(2 * np.pi * hr / 24)
    out["mnth_sin"] = np.sin(2 * np.pi * (out["mnth"] - 1) / 12)
    out["mnth_cos"] = np.cos(2 * np.pi * (out["mnth"] - 1) / 12)
    out["temp_sq"] = out["temp"] ** 2
    out["day_type"] = np.where(out["workingday"] == 1, "Working day", "Non-working day")
    return out


def daily_peak_share(hour_df: pd.DataFrame, min_hours: int = 22) -> pd.DataFrame:
    """Per day: share of rentals in the commute windows (07-09, 16-19), for every day type.

    Uses clock-hour windows (not the working-day-only flags) so working and non-working days
    are measured on the same 7 hours. Days with < `min_hours` recorded hours are dropped,
    because a partial day distorts the share.
    """
    g = hour_df.assign(in_peak=hour_df["hr"].isin(PEAK_HOURS) * hour_df["cnt"])
    daily = g.groupby("dteday").agg(
        cnt=("cnt", "sum"), peak_cnt=("in_peak", "sum"), n_hours=("hr", "size"),
        workingday=("workingday", "first"), yr=("yr", "first"), season=("season", "first"),
        mnth=("mnth", "first"), weather_mean=("weather", "mean"), temp=("temp", "mean"),
        hum=("hum", "mean"), windspeed=("windspeed", "mean"),
    ).reset_index()
    daily = daily[daily["n_hours"] >= min_hours].copy()
    daily["peak_share"] = daily["peak_cnt"] / daily["cnt"]
    return daily


In [ ]:
%%writefile src/hypothesis_testing.py
"""
Statistical tests used in Phase 2.

* compare_groups(): Welch two-sample t-test + assumption checks + effect size + non-parametric check.
* ols_hac(): OLS with Newey-West (HAC) standard errors, written with NumPy only, used to
  (a) control for confounders (season, weather, temperature, year) and
  (b) allow for day-to-day autocorrelation, which violates the t-test's independence assumption.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats


def compare_groups(a: pd.Series, b: pd.Series, alpha: float = 0.05,
                   label_a: str = "group A", label_b: str = "group B") -> dict:
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    ma, mb = a.mean(), b.mean()
    va, vb = a.var(ddof=1), b.var(ddof=1)

    t = stats.ttest_ind(a, b, equal_var=False)
    se = np.sqrt(va / na + vb / nb)
    df = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    tcrit = stats.t.ppf(1 - alpha / 2, df)
    diff = ma - mb

    pooled_sd = np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))
    d = diff / pooled_sd
    hedges_g = d * (1 - 3 / (4 * (na + nb) - 9))

    mw = stats.mannwhitneyu(a, b, alternative="two-sided")
    rank_biserial = 1 - 2 * mw.statistic / (na * nb)   # sign: positive => A tends to be larger

    return {
        "label_a": label_a, "label_b": label_b,
        "n_a": na, "n_b": nb, "mean_a": ma, "mean_b": mb,
        "sd_a": np.sqrt(va), "sd_b": np.sqrt(vb),
        "diff": diff, "ci_low": diff - tcrit * se, "ci_high": diff + tcrit * se,
        "welch_t": t.statistic, "welch_df": df, "p_value": t.pvalue,
        "hedges_g": hedges_g,
        "levene_p": stats.levene(a, b).pvalue,
        "shapiro_p_a": stats.shapiro(a).pvalue, "shapiro_p_b": stats.shapiro(b).pvalue,
        "skew_a": stats.skew(a), "skew_b": stats.skew(b),
        "mannwhitney_p": mw.pvalue, "rank_biserial": -rank_biserial,
        "reject_h0": t.pvalue < alpha, "alpha": alpha,
    }


def format_result(r: dict, value_fmt: str = "{:.3f}") -> pd.DataFrame:
    f = lambda v: value_fmt.format(v)
    rows = [
        (f"n ({r['label_a']} / {r['label_b']})", f"{r['n_a']} / {r['n_b']}"),
        (f"Mean {r['label_a']}", f(r["mean_a"])),
        (f"Mean {r['label_b']}", f(r["mean_b"])),
        ("Difference (A - B)", f(r["diff"])),
        ("95% CI of difference", f"[{f(r['ci_low'])}, {f(r['ci_high'])}]"),
        ("Welch t (df)", f"{r['welch_t']:.2f} ({r['welch_df']:.0f})"),
        ("p-value (Welch, two-sided)", f"{r['p_value']:.2e}"),
        ("Effect size Hedges' g", f"{r['hedges_g']:.2f}"),
        ("Levene p (equal variances?)", f"{r['levene_p']:.3g}"),
        ("Shapiro p (A / B normal?)", f"{r['shapiro_p_a']:.3g} / {r['shapiro_p_b']:.3g}"),
        ("Mann-Whitney U p (robustness)", f"{r['mannwhitney_p']:.2e}"),
        ("Rank-biserial correlation", f"{r['rank_biserial']:.2f}"),
        (f"Decision at alpha = {r['alpha']}", "Reject H0" if r["reject_h0"] else "Fail to reject H0"),
    ]
    return pd.DataFrame(rows, columns=["Statistic", "Value"]).set_index("Statistic")


def ols_hac(y: pd.Series, X: pd.DataFrame, maxlags: int = 7) -> pd.DataFrame:
    """OLS with Newey-West standard errors (Bartlett kernel). Rows must be in time order."""
    Xc = np.column_stack([np.ones(len(X)), X.to_numpy(float)])
    names = ["const"] + list(X.columns)
    yv = np.asarray(y, float)
    XtX_inv = np.linalg.inv(Xc.T @ Xc)
    beta = XtX_inv @ Xc.T @ yv
    u = yv - Xc @ beta
    Xu = Xc * u[:, None]
    S = Xu.T @ Xu
    for lag in range(1, maxlags + 1):
        w = 1 - lag / (maxlags + 1)
        G = Xu[lag:].T @ Xu[:-lag]
        S += w * (G + G.T)
    n, k = Xc.shape
    cov = XtX_inv @ S @ XtX_inv * n / (n - k)
    se = np.sqrt(np.diag(cov))
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), n - k)
    r2 = 1 - (u @ u) / ((yv - yv.mean()) @ (yv - yv.mean()))
    out = pd.DataFrame({"coef": beta, "hac_se": se, "t": t, "p_value": p,
                        "ci_low": beta - 1.96 * se, "ci_high": beta + 1.96 * se}, index=names)
    out.attrs["r2"] = r2
    out.attrs["n"] = n
    return out


In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

sys.path.insert(0, "src")
import preprocessing as pp, feature_engineering as fe, hypothesis_testing as ht
for m in (pp, fe, ht):
    importlib.reload(m)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning); warnings.filterwarnings("ignore", message=".*vert.*")

# Chart style: one consistent, colour-blind-checked palette
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
INK2, GRID, SURFACE = "#52514e", "#e6e5e0", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#b9b8b0", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "legend.frameon": False,
})
DAYTYPE_COLOR = {"Working day": BLUE, "Non-working day": ORANGE}

def save(fig, name):
    fig.savefig(f"outputs/charts/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 1. Data quality – apply the cleaning decisions
Nothing is deleted. Each correction is **flagged** in a new column so it stays visible.

In [ ]:
hour_raw, day_raw = pp.load_hour("data/hour.csv"), pp.load_day("data/day.csv")
hour = pp.clean_hour(hour_raw)
day = pp.clean_day(day_raw)

display(pd.DataFrame(pp.CLEANING_LOG, columns=["Issue", "Action"]))

print("Humidity imputed (hourly rows):", hour.hum_imputed.sum(), "| new values on 2011-03-10:",
      hour.loc[hour.hum_imputed == 1, "hum"].round(2).agg(["min", "max"]).to_dict())
print("Humidity imputed (daily rows):", day.hum_imputed.sum(), "| new value:", day.loc[day.hum_imputed == 1, "hum"].round(3).tolist())
print("Remaining hum == 0:", (hour.hum == 0).sum())
print("weather categories after merge:", hour.weather.value_counts().sort_index().to_dict())
print("Rows on disrupted days:", hour.disrupted_day.sum())
print("Raw files unchanged:", hour_raw.shape, day_raw.shape)

**Result.** The 22 impossible humidity values on 2011-03-10 now range 0.62–0.83, which is normal for March (daily value 0.712). Weather category 3 now holds 1,422 hours (3 merged in). 101 hourly rows sit on the 8 disrupted days and are flagged, not dropped. Raw files are untouched.

> **If asked "why impute instead of dropping the day?"** – Dropping 2011-03-10 would discard 22 hours of *real rental counts* because one weather sensor failed. Imputing from the same hour on neighbouring days is a small, transparent fix and it is flagged so it can be excluded in a sensitivity check.

## 2. Feature engineering
Rule: a feature must be **known before the hour starts** – calendar facts, plus weather (which in deployment comes from a forecast).
No `casual`, `registered`, or future rows. No lag features in this phase; whether a "same hour last week" lag adds value is tested in Phase 3.

In [ ]:
hour = fe.add_features(hour)
day = fe.add_features(day)

display(pd.DataFrame.from_dict(fe.FEATURE_RATIONALE, orient="index", columns=["Business reason"]))
display(pd.DataFrame.from_dict(fe.EXCLUDED_COLUMNS, orient="index", columns=["Why excluded from model inputs"]))

print("Model feature set (", len(fe.MODEL_FEATURES), "):", fe.MODEL_FEATURES)
display(hour[["timestamp", "hr", "workingday", "rush_hour", "morning_peak", "evening_peak", "night",
              "hr_sin", "hr_cos", "weather", "temp", "temp_sq", "cnt"]].iloc[5:11].round({"hr_sin": 3, "hr_cos": 3, "temp_sq": 3}))

# Sanity checks on engineered flags
assert hour.loc[hour.rush_hour == 1, "workingday"].eq(1).all()
assert np.allclose(hour.hr_sin**2 + hour.hr_cos**2, 1)
print("Rush-hour share of all hours:", f"{hour.rush_hour.mean():.1%}",
      "| share of all rentals in rush hours:", f"{hour.loc[hour.rush_hour == 1, 'cnt'].sum() / hour.cnt.sum():.1%}")

**Result.** Working-day rush hours are **20% of all hours but carry 40.3% of all rentals** – the first quantitative sign that demand is concentrated.

Notes to remember:
- `workingday`, `is_weekend` and `holiday` are linearly dependent (every holiday in this data falls on Mon–Fri, so working day = not weekend and not holiday). Tree models don't mind; for the **linear model in Phase 3 one of them will be dropped** to avoid perfect collinearity.
- `hr` is kept as a plain number for tree models (they can split 7–9 vs 10–15 directly). The sine/cosine versions exist for the linear model.

> **If asked "why sine/cosine?"** – As a plain number, hour 23 and hour 0 look 23 units apart; on a circle they are neighbours. **"Why no lag features?"** – They need the previous week's actual demand, which limits what-if questions in the prototype. I test whether they add accuracy in Phase 3 instead of assuming it.

## 3. Exploratory data analysis
Each chart answers a business question. Numbers printed under each chart are the ones quoted in the interpretation.

### 3.1 Distribution of hourly demand

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(hour.cnt, bins=50, color=BLUE, edgecolor=SURFACE, linewidth=0.6)
ax[0].set(title="Hourly rentals (cnt)", xlabel="rentals per hour", ylabel="number of hours")
ax[1].hist(np.log1p(hour.cnt), bins=50, color=BLUE, edgecolor=SURFACE, linewidth=0.6)
ax[1].set(title="log(1 + cnt)", xlabel="log(1 + rentals per hour)")
plt.tight_layout(); save(fig, "01_distribution")

q = hour.cnt.quantile([.1, .25, .5, .75, .9, .95, .99])
print("mean", round(hour.cnt.mean(), 1), "| skew", round(hour.cnt.skew(), 2), "| log1p skew", round(np.log1p(hour.cnt).skew(), 2))
print(q.round(0).to_dict())
print("Hours with < 20 rentals:", f"{(hour.cnt < 20).mean():.1%}", "| hours with > 500:", f"{(hour.cnt > 500).mean():.1%}")

**What it shows.** Hourly demand is strongly right-skewed: median **142** vs mean **189.5**; 95th percentile **563**, 99th **782**. **16.7% of hours have fewer than 20 rentals; only 7.4% exceed 500.** `log(1+cnt)` over-corrects (skew −0.82), so a log target is something to *test* in Phase 3, not assume.

**Business insight.** Most hours are quiet and a small number of hours are very busy. A fleet sized to cover the busiest hours will, by construction, be partly idle most of the day – which is exactly the CEO's observation.

> **If asked "are the high values outliers?"** – They are the busiest real hours (82% on working days at 8 AM / 5–6 PM). Removing them would remove the problem we're asked to solve.

### 3.2 Hourly demand pattern – working vs non-working days

In [ ]:
prof = hour.groupby(["day_type", "hr"]).cnt.agg(["mean", lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
prof.columns = ["mean", "q25", "q75"]

fig, ax = plt.subplots(figsize=(11, 4.2))
for dt, c in DAYTYPE_COLOR.items():
    p = prof.loc[dt]
    ax.fill_between(p.index, p.q25, p.q75, color=c, alpha=.15, linewidth=0)
    ax.plot(p.index, p["mean"], color=c, marker="o", markersize=4, label=f"{dt} (mean, band = middle 50%)")
ax.axvspan(6.5, 9.5, color=GRID, alpha=.5, zorder=0); ax.axvspan(15.5, 19.5, color=GRID, alpha=.5, zorder=0)
ax.text(8, ax.get_ylim()[1]*.95, "morning peak", ha="center", color=INK2); ax.text(17.5, ax.get_ylim()[1]*.95, "evening peak", ha="center", color=INK2)
ax.set(title="Average rentals by hour of day", xlabel="hour of day", ylabel="rentals per hour", xticks=range(24))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2); save(fig, "02_hourly_pattern")

m = prof["mean"].unstack(0)
print(m.loc[[3, 7, 8, 9, 12, 13, 17, 18, 19]].round(0))
for dt in DAYTYPE_COLOR:
    s = m[dt]; print(f"{dt}: peak {s.idxmax()}:00 = {s.max():.0f}, trough {s.idxmin()}:00 = {s.min():.0f}, "
                     f"peak / daily-average-hour = {s.max() / s.mean():.2f}x")

**What it shows (the most important chart).**
- **Working days:** two sharp peaks – **8:00 ≈ 477** and **17:00 ≈ 525** (18:00 ≈ 492) – with a midday dip to ≈ 200 and near-zero at night (3:00 ≈ 5). Busiest hour = **2.74×** the average hour. The middle-50% band at 17:00 reaches ~700.
- **Non-working days:** one broad midday plateau, peak **13:00 ≈ 373**, busiest hour = **2.06×** the average hour.

**Business insight.** The pressure on availability comes from roughly **7 hours on working days** (07–09, 16–19). Between them, at midday, demand halves; overnight it almost disappears. A commute peak also implies *direction* – mornings typically move bikes from residential areas into business areas and evenings move them back – so bikes "pile up" at destinations while origins empty.

**Why it matters operationally.** If shortages happen, they are most likely in these windows, and the quiet midday and night hours are when rebalancing trucks can reposition bikes. *Caveat:* this is a city-wide total; the data has no station information, so the direction of flows is an inference to be confirmed with trip-level data.

### 3.3 Demand across weekdays (daily totals)

In [ ]:
order = [1, 2, 3, 4, 5, 6, 0]
names = {0: "Sun", 1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat"}
wk = day.groupby("weekday").cnt.agg(["mean", "std", "count"]).loc[order]
wk["ci95"] = 1.96 * wk["std"] / np.sqrt(wk["count"])

fig, ax = plt.subplots(figsize=(9, 3.8))
colors = [BLUE if d in (1, 2, 3, 4, 5) else ORANGE for d in order]
ax.bar([names[d] for d in order], wk["mean"], yerr=wk["ci95"], color=colors, edgecolor=SURFACE, linewidth=2,
       capsize=4, error_kw={"ecolor": INK2, "lw": 1})
ax.set(title="Average daily rentals by weekday (error bar = 95% CI)", ylabel="rentals per day")
from matplotlib.patches import Patch
ax.set_ylim(0, 6200)
ax.legend(handles=[Patch(color=BLUE, label="Mon-Fri"), Patch(color=ORANGE, label="Sat-Sun")], loc="upper center", ncol=2)
save(fig, "03_weekday")

print(wk.assign(day=[names[d] for d in order]).set_index("day")[["mean", "ci95", "count"]].round(0))

**What it shows.** Average daily rentals range only from **4,229 (Sun)** to **4,690 (Fri)**; the 95% confidence intervals (±340–420) overlap for every pair of days.

**Business insight.** *Which* weekday it is barely changes daily volume. What changes is the hourly *shape* (chart 3.2). Planning should be by **day type** (working vs non-working), not by individual weekday.

### 3.4 Working-day vs non-working-day daily demand (split by year)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
data, pos, cols = [], [], []
for i, yr in enumerate([0, 1]):
    for j, dt in enumerate(["Working day", "Non-working day"]):
        data.append(day.loc[(day.yr == yr) & (day.day_type == dt), "cnt"]); pos.append(i * 3 + j); cols.append(DAYTYPE_COLOR[dt])
bp = ax.boxplot(data, positions=pos, widths=.7, patch_artist=True, medianprops={"color": "white", "lw": 2})
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c); patch.set_edgecolor(c)
ax.set_xticks([.5, 3.5]); ax.set_xticklabels(["2011", "2012"])
ax.set(title="Daily rentals: working vs non-working days", ylabel="rentals per day", ylim=(0, 10500))
ax.legend(handles=[Patch(color=c, label=k) for k, c in DAYTYPE_COLOR.items()], loc="upper left")
save(fig, "04_working_vs_nonworking")

print(day.groupby(["yr", "day_type"]).cnt.agg(["count", "mean", "median", "std"]).round(0))

**What it shows.** In 2011, working and non-working days have almost the same mean (**3,425 vs 3,364**); in 2012 the gap is larger (**5,745 vs 5,288**). The jump between years (~+2,000/day) is far bigger than any day-type difference. Non-working days are more spread out (SD 2,048 in 2012) – leisure riding swings more with weather.

**Why split by year?** Pooling both years mixes the system's growth into every comparison. Year is a confounder that must be controlled (done in 4.4–4.5).

### 3.5 Monthly and seasonal demand

In [ ]:
mon = day.groupby(["yr", "mnth"]).cnt.mean().unstack(0)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [2, 1]})
ax[0].plot(mon.index, mon[0], color=BLUE, marker="o", label="2011")
ax[0].plot(mon.index, mon[1], color=ORANGE, marker="o", label="2012")
ax[0].set(title="Average daily rentals by month", xlabel="month", ylabel="rentals per day", xticks=range(1, 13))
ax[0].legend()
sea = day.groupby("season").cnt.mean()
ax[1].bar(["Spring", "Summer", "Fall", "Winter"], sea.values, color=BLUE, edgecolor=SURFACE, linewidth=2)
ax[1].set(title="By season (both years)", ylabel="rentals per day")
plt.tight_layout(); save(fig, "05_monthly_seasonal")

print(mon.rename(columns={0: "2011", 1: "2012"}).round(0).T)
print("Season means:", sea.round(0).to_dict(), " (1=spring ... 4=winter)")
print("Peak month / lowest month ratio:", (mon.max() / mon.min()).round(2).to_dict())

**What it shows.** Strong seasonality: 2011 goes from **1,232/day in January to 4,784 in June (3.9×)**; 2012 from **3,121 in January to 7,286 in September (2.3×)**. By season, Fall is highest (5,644) and "Spring" lowest (2,604).

**Careful with the labels:** in this dataset season 1 ("spring") runs roughly 21 Dec – 20 Mar, i.e. the coldest months. That is why "spring" is lowest – it isn't a data error.

**Business insight.** The number of bikes needed in January is a fraction of what is needed in September. Buying bikes for the summer/autumn peak means even more idle bikes in winter. Seasonal levers (winter maintenance, temporary fleet reduction, short-term leasing for peak months) are cheaper than a permanent purchase sized for the peak.

### 3.6 Demand under different weather conditions

In [ ]:
wnames = {1: "Clear", 2: "Mist/cloud", 3: "Rain/snow"}
wcol = {1: BLUE, 2: AQUA, 3: ORANGE}
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 2]})
wm = hour.groupby("weather").cnt.agg(["mean", "count"])
ax[0].bar([wnames[w] for w in wm.index], wm["mean"], color=[wcol[w] for w in wm.index], edgecolor=SURFACE, linewidth=2)
for i, (w, r) in enumerate(wm.iterrows()):
    ax[0].text(i, r["mean"] + 4, f"n={int(r['count']):,}", ha="center", color=INK2, fontsize=9)
ax[0].set(title="Average rentals per hour", ylabel="rentals per hour", ylim=(0, wm["mean"].max() * 1.18))
wp = hour[hour.workingday == 1].groupby(["weather", "hr"]).cnt.mean().unstack(0)
for w in wp.columns:
    ax[1].plot(wp.index, wp[w], color=wcol[w], marker="o", markersize=3, label=wnames[w])
ax[1].set(title="Working days: hourly pattern by weather", xlabel="hour of day", ylabel="rentals per hour", xticks=range(24))
ax[1].legend(); plt.tight_layout(); save(fig, "06_weather")

rush = hour[hour.rush_hour == 1].groupby("weather").cnt.mean()
print("All hours:", wm["mean"].round(0).to_dict())
print("Working-day rush hours:", rush.round(0).to_dict(),
      f"| rain vs clear: {rush[3] / rush[1] - 1:+.0%}, mist vs clear: {rush[2] / rush[1] - 1:+.0%}")
print("Share of hours with rain/snow:", f"{(hour.weather == 3).mean():.1%}")

**What it shows.** Average rentals per hour: **clear 205, mist/cloud 175, rain/snow 112**. Comparing like with like (working-day rush hours only, which removes the time-of-day effect): **clear 414, mist 358 (−14%), rain/snow 226 (−45%)**. The *shape* of the day stays the same – the commute peaks still exist in the rain, just lower. Rain/snow covers only 8.2% of hours.

**Business insight.** Weather mainly scales demand up or down; it doesn't move the peaks. On a rainy working day, roughly half the usual bikes are needed at the peaks, which frees trucks and staff for maintenance. The daily operating plan should use the weather forecast.

### 3.7–3.9 Temperature, humidity and wind speed vs demand
Binned averages (bins with at least 30 hours) are easier to read than a 17,000-point scatter.

In [ ]:
def binned(col, bins):
    b = pd.cut(hour[col], bins)
    g = hour.groupby(b, observed=True).cnt.agg(["mean", "count", lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
    g.columns = ["mean", "count", "q25", "q75"]
    g = g[g["count"] >= 30]
    g.index = [iv.mid for iv in g.index]
    return g

specs = [("temp_c", np.arange(0, 44, 2), "Temperature (°C)", "07_temperature"),
         ("hum_pct", np.arange(0, 105, 5), "Humidity (%)", "08_humidity"),
         ("wind_unscaled", np.arange(0, 60, 3), "Wind speed (Readme: value x 67, unit not stated)", "09_windspeed")]
from scipy.stats import spearmanr
for col, bins, label, fname in specs:
    g = binned(col, bins)
    fig, ax = plt.subplots(figsize=(8, 3.3))
    ax.fill_between(g.index, g.q25, g.q75, color=BLUE, alpha=.15, linewidth=0, label="middle 50% of hours")
    ax.plot(g.index, g["mean"], color=BLUE, marker="o", markersize=4, label="average")
    ax.set(title=f"Rentals per hour vs {label.split(' (')[0].lower()}", xlabel=label, ylabel="rentals per hour")
    ax.legend(loc="upper left"); save(fig, fname)
    rho = spearmanr(hour[col], hour.cnt).statistic
    print(f"{label}: Spearman rho with cnt = {rho:+.2f}; highest bin avg {g['mean'].max():.0f} at {g['mean'].idxmax():.1f}, "
          f"lowest {g['mean'].min():.0f} at {g['mean'].idxmin():.1f}")

# Temperature is confounded with season and hour: check within working-day 17:00 only
sub = hour[(hour.workingday == 1) & (hour.hr == 17)]
print("Working-day 17:00 only -> Spearman(temp, cnt) =", round(spearmanr(sub.temp, sub.cnt).statistic, 2),
      "| Spearman(hum, cnt) =", round(spearmanr(sub.hum, sub.cnt).statistic, 2))
print("Humidity vs weather: mean hum by weather", hour.groupby("weather").hum_pct.mean().round(0).to_dict())

**Temperature** (Spearman ρ = **+0.42**). Demand rises from ≈ 35/hr around 3 °C to ≈ 344/hr around 33 °C, then drops at the hottest temperatures (~39 °C, few hours) – the "too hot to ride" effect that the `temp_sq` feature captures. Looking only at working days at 17:00 (same hour and day type), ρ is still **0.60**, so this is not just a side effect of season or time of day.

**Humidity** (ρ = **−0.36**). Demand falls steadily from ≈ 322/hr at low humidity to ≈ 69/hr near 100%. Part of this is rain and night-time in disguise: average humidity is 57% in clear hours, 70% in mist and 84% in rain. Within working-day 17:00 the effect shrinks to ρ = −0.24.

**Wind speed** (ρ = **+0.13**). Essentially flat between ~10 and 35 units; the weak positive sign likely reflects windier daytime hours, not wind *causing* riding. (Hours coded exactly 0 – 12.5% of rows – fall outside the first bin and are not plotted.)

**Business insight.** After rain, temperature is the weather variable that matters; wind adds little. These are associations: weather moves with season and time of day, which is why the model and the hypothesis regression include those as controls.

### 3.10 Demand trend over time

In [ ]:
s = day.set_index("dteday").cnt
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.plot(s.index, s, color=BLUE, alpha=.35, linewidth=1, label="daily rentals")
ax.plot(s.index, s.rolling(30, center=True).mean(), color=BLUE, linewidth=2.5, label="30-day rolling mean")
dis = day[day.disrupted_day == 1]
ax.scatter(dis.dteday, dis.cnt, color=RED, s=40, zorder=3, edgecolor=SURFACE, linewidth=1.5, label="disrupted day (storm / partial data)")
ax.annotate("Hurricane Sandy", (pd.Timestamp("2012-10-29"), s["2012-10-29"]), xytext=(-110, 10), textcoords="offset points",
            color=INK2, arrowprops={"arrowstyle": "-", "color": INK2})
ax.set(title="Daily rentals, 2011–2012", ylabel="rentals per day"); ax.legend(loc="upper left")
save(fig, "10_trend")

r30 = s.rolling(30).mean()
print("Avg daily rentals: 2011 =", round(s[:"2011"].mean()), "| 2012 =", round(s["2012":].mean()),
      f"| growth {s['2012':].mean() / s[:'2011'].mean() - 1:+.0%}")
print("Same-month growth (2012 vs 2011):", (mon[1] / mon[0] - 1).round(2).to_dict())
print("Disrupted days:", dis[["dteday", "cnt"]].assign(dteday=dis.dteday.dt.date).to_dict("records"))

**What it shows.** Average daily rentals were **3,406 in 2011 and 5,600 in 2012 (+64%)**. Growth was uneven – early-2012 months more than doubled (Jan +153%, Mar +157%) while later months grew 41–72%. Demand dips sharply on the disrupted days (Hurricane Sandy: **22 rentals** on 2012-10-29).

**Business insight.** The system was growing fast. Whether to buy bikes for the future depends on whether this growth continues – something two years of data cannot forecast reliably.

**Modelling consequence.** Demand is not stationary. A model trained only on 2011 would under-predict 2012; this is why the Phase 3 training window includes early 2012 and the model gets `yr` as a feature.

### 3.11 Correlation heatmap

In [ ]:
cols = ["cnt", "registered", "casual", "temp", "atemp", "hum", "windspeed", "weather", "hr", "workingday", "yr", "mnth", "season"]
corr = hour[cols].corr()
cmap = LinearSegmentedColormap.from_list("div", [BLUE, "#f0efec", RED])
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right"); ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
ax.grid(False)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > .6 else "#0b0b0b")
fig.colorbar(im, ax=ax, shrink=.8, label="Pearson correlation")
ax.set_title("Correlation matrix (hourly data)"); save(fig, "11_correlation")

print("Correlation with cnt:\n", corr["cnt"].drop("cnt").sort_values(key=abs, ascending=False).round(2).to_string())
print("\ntemp vs atemp:", round(corr.loc["temp", "atemp"], 3), "| season vs mnth:", round(corr.loc["season", "mnth"], 2))

**What it shows.**
- `registered` (0.97) and `casual` (0.69) correlate most with `cnt` – because they **are** `cnt`. This is the leakage, made visible.
- `temp` vs `atemp` = **0.988** → near-duplicates; keep `temp` only.
- `mnth` vs `season` = 0.83 → overlapping information (fine for trees; redundant for linear models).
- `hr` correlates only **0.39** and `workingday` only **0.03** with demand – yet chart 3.2 shows they are the two biggest drivers.

**Key lesson (good interview point).** Pearson correlation only measures *straight-line* relationships. Hour has two peaks, and working day changes the *shape* of the day rather than the total, so both look weak here. This is why we don't drop features based on low correlation, and why we need cyclical encodings, interactions, or tree models.

### 3.12 Registered vs casual riders

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for k, dt in enumerate(["Working day", "Non-working day"]):
    p = hour[hour.day_type == dt].groupby("hr")[["registered", "casual"]].mean()
    ax[k].plot(p.index, p.registered, color=BLUE, marker="o", markersize=3, label="registered")
    ax[k].plot(p.index, p.casual, color=ORANGE, marker="o", markersize=3, label="casual")
    ax[k].set(title=dt, xlabel="hour of day", xticks=range(0, 24, 2))
ax[0].set_ylabel("rentals per hour"); ax[0].legend(loc="upper left")
plt.tight_layout(); save(fig, "12_registered_vs_casual")

sh = hour.groupby("day_type")[["casual", "registered", "cnt"]].sum()
print("Casual share of rentals:", (sh.casual / sh.cnt).round(3).to_dict())
pk = hour[hour.hr.isin(fe.PEAK_HOURS)].groupby("day_type")[["casual", "registered"]].sum() / hour.groupby("day_type")[["casual", "registered"]].sum()
print("Share of each group's rentals that fall in 07-09 & 16-19:\n", pk.round(3))
print("Correlation of casual with temp:", round(hour.casual.corr(hour.temp), 2), "| registered with temp:", round(hour.registered.corr(hour.temp), 2))

**What it shows.** Registered members follow the commute pattern: on working days **60.3%** of their rentals fall in the 7 peak hours. Casual riders follow a leisure curve (afternoon, weekends): **41.9%** in those hours on working days. Casual riders are **13.2%** of working-day rentals but **31.7%** of non-working-day rentals, and are more temperature-sensitive (correlation with temp 0.46 vs 0.34).

**Business insight.** The commute-peak shortages, if they happen, hit **registered members** – the recurring, subscription customers whose complaints matter most for retention. Casual demand is a separate, weather-driven weekend/afternoon market, likely around leisure locations.

*(Registered/casual counts are used here only to understand behaviour; they are never model inputs.)*

### 3.13 EDA summary – what this means for the CEO's question

In [ ]:
wd = hour[(hour.workingday == 1) & (hour.disrupted_day == 0)]
per_day = wd.groupby("dteday").cnt.agg(["max", "mean", "sum"])
per_day["peak_to_mean"] = per_day["max"] / per_day["mean"]
night_share = hour.loc[hour.night == 1, "cnt"].sum() / hour.cnt.sum()
print(f"Working days: median peak hour / average hour = {per_day.peak_to_mean.median():.2f}x")
print(f"Working days: median share of the day's rentals in the 7 peak hours = "
      f"{(wd[wd.hr.isin(fe.PEAK_HOURS)].groupby('dteday').cnt.sum() / per_day['sum']).median():.1%} (uniform would be {7/24:.1%})")
print(f"Share of all rentals between 00:00 and 05:59: {night_share:.1%}")

**Putting the EDA together:**
- On a typical working day, the busiest hour is **2.9×** the average hour, and **58.6%** of the day's rentals happen in 7 hours (an even spread would be 29.2%).
- Only **3.2%** of rentals happen between midnight and 6 AM.
- Demand varies **2–4×** across months and drops **~45%** in rain at the same hours.
- The system grew **+64%** in a year.

**What this says about the CEO's question.** The data is *consistent with* a timing-and-placement problem: demand is squeezed into short windows and moves in one direction at a time, so the same fleet is too small at 8 AM in some places and idle at noon. It does **not prove** there are shortages – the data has only completed rentals, not failed attempts or empty stations. That is what the hypothesis below quantifies and what the experiment in 4.6 would confirm.

## 4. Hypothesis testing

### 4.1 Choosing the hypothesis (exploration on 2011 only)
To avoid "finding a hypothesis in the data and then testing it on the same data", the hypothesis is **formed on 2011** and **confirmed on 2012**, which it has not seen.
(2012 H2 is also the Phase 3 model test set. The hypothesis test fits no model, so this does not leak into model evaluation.)

In [ ]:
daily = fe.daily_peak_share(hour, min_hours=22)        # drops partial/storm days
print("Days kept:", len(daily), "of", hour.dteday.nunique(), "(partial days removed: share would be distorted)")

explore = daily[daily.yr == 0]
fig, ax = plt.subplots(figsize=(9, 3.6))
bins = np.linspace(0, .8, 41)
for wdv, label in [(1, "Working day"), (0, "Non-working day")]:
    ax.hist(explore.loc[explore.workingday == wdv, "peak_share"], bins=bins, color=DAYTYPE_COLOR[label], alpha=.65, label=label, edgecolor=SURFACE)
ax.axvline(7 / 24, color=INK2, ls="--", lw=1); ax.text(7 / 24 + .005, ax.get_ylim()[1] * .9, "uniform = 29%", color=INK2)
ax.set(title="2011 (exploration): share of daily rentals in 07-09 & 16-19", xlabel="peak share", ylabel="days"); ax.legend()
save(fig, "13_peak_share_2011")
print(explore.groupby("workingday").peak_share.describe().round(3))
print("\nFor contrast – daily TOTAL rentals 2011 by day type:", explore.groupby("workingday").cnt.mean().round(0).to_dict())

**What 2011 suggested.** Working days put **57.9%** of rentals into the commute windows (SD 0.042); non-working days only **34.1%** (SD 0.042). The two distributions barely overlap – while daily *totals* in 2011 are almost the same (3,468 vs 3,375). That contrast – *same volume, different timing* – is what motivated the hypothesis.

### 4.2 The hypothesis
**Metric:** *peak share* = rentals in the commute windows (07:00–09:59 and 16:00–19:59) ÷ the day's total rentals. One value per day.

- **H0:** mean peak share is the same on working days and non-working days.
- **H1:** mean peak share differs between working and non-working days (two-sided; 2011 suggests working days are higher).
- **α = 0.05.** Unit of analysis = one day. Confirmation sample = **2012** days with ≥ 22 recorded hours.

**Why this hypothesis and not "daily demand differs"?** The CEO's puzzle – *riders can't find bikes, yet bikes sit idle* – is exactly what you'd see if demand is fine in **total** but crowded into **a few hours**. A peak-share test checks that mechanism directly. The suggested daily-total test is run as well (4.5).

**Test choice:** Welch's t-test (does not assume equal variances; groups have unequal size ~250 vs ~115). With n > 100 per group the sampling distribution of the mean is close to normal even if days are not, but normality and variances are checked and a Mann–Whitney U test is reported as a robustness check.

In [ ]:
confirm = daily[daily.yr == 1]
res = ht.compare_groups(confirm.loc[confirm.workingday == 1, "peak_share"], confirm.loc[confirm.workingday == 0, "peak_share"],
                        label_a="working", label_b="non-working")
display(ht.format_result(res))
pd.DataFrame([res]).to_csv("outputs/model_results/hypothesis_peak_share_2012.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 3.4))
bp = ax.boxplot([confirm.loc[confirm.workingday == w, "peak_share"] for w in (1, 0)], vert=False, widths=.6,
                patch_artist=True, medianprops={"color": "white", "lw": 2})
for patch, c in zip(bp["boxes"], [BLUE, ORANGE]):
    patch.set_facecolor(c); patch.set_edgecolor(c)
ax.set_yticks([1, 2]); ax.set_yticklabels(["Working day", "Non-working day"])
ax.axvline(7 / 24, color=INK2, ls="--", lw=1)
ax.set(title="2012 (confirmation): daily peak share", xlabel="share of daily rentals in 07-09 & 16-19")
save(fig, "14_peak_share_2012_test")

**Results on 2012 (not used to form the hypothesis):**

| | |
|---|---|
| Mean peak share | **working 0.581** vs **non-working 0.343** (n = 248 / 116 days) |
| Difference | **+23.8 percentage points**, 95% CI **[22.9, 24.8]** |
| Welch t | 47.8 (df 224), **p ≈ 1.7 × 10⁻¹¹⁹** |
| Effect size | Hedges' g = **5.38** (anything above 0.8 is "large") |
| Assumptions | Levene p = 0.79 (variances similar; Welch is safe either way). Shapiro rejects normality, but with 116+ days per group the mean is approximately normal (CLT) |
| Robustness | Mann–Whitney U p ≈ 1.1 × 10⁻⁵¹, rank-biserial 0.98 – same conclusion without the normality assumption |
| **Decision** | **Reject H0** at α = 0.05 |

**Business interpretation.** On working days, **about 58% of rentals are squeezed into 7 hours**, compared with 34% on non-working days. The direction is unsurprising (people commute); what matters is the **size**: availability on working days is decided by a few hours, not by daily volume.

> **If an interviewer says "that's obvious":** Agreed on the direction – the value is in quantifying it and showing it holds out-of-sample and after controls. It changes the question from *"how many rides per day?"* to *"how many bikes in the right place between 7–9 and 16–19?"* – and that is what drives the buy-vs-rebalance decision.

### 4.3 Assumptions and confounders
- **Independence:** consecutive days are correlated (weather spells, trend). The t-test treats days as independent, so its p-value is somewhat optimistic. → handled in 4.4 with autocorrelation-robust (Newey–West) standard errors.
- **Confounding:** working and non-working days could differ in season, weather or year mix (e.g. holidays cluster in winter). → controlled in 4.4 by regression.
- **Causation:** even after controls, this shows *association*. The working-day effect is plausibly causal (commuting), but the test itself does not prove it.

In [ ]:
X = pd.get_dummies(daily[["workingday", "yr", "season", "weather_mean", "temp", "hum", "windspeed"]],
                   columns=["season"], drop_first=True, dtype=float)
reg = ht.ols_hac(daily.peak_share, X, maxlags=7)
display(reg.round(4))
print(f"R² = {reg.attrs['r2']:.3f}, n = {reg.attrs['n']} days, Newey-West lags = 7")
print("Day-type mix check (share of each season among working / non-working days):")
print(pd.crosstab(daily.workingday, daily.season, normalize="index").round(2))
reg.to_csv("outputs/model_results/hypothesis_peak_share_regression.csv")

**Result.** After controlling for year, season, weather, temperature, humidity and wind – with Newey–West standard errors that allow for correlation between nearby days – the working-day coefficient is **+0.238 (95% CI 0.231 to 0.246)**, practically identical to the raw difference. None of the controls is significant (all p > 0.15), and the season mix is similar for both day types (23–27% each).

**Interpretation.** The concentration of demand in commute windows is a stable property of working days, not a side effect of weather, season or growth. Weather lowers demand in *all* hours roughly proportionally, so it doesn't change the share. This is still an *association*; the plausible mechanism is commuting.

### 4.5 The suggested test – do daily *totals* differ? (day.csv, 2011–2012)

In [ ]:
res_tot = ht.compare_groups(day.loc[day.workingday == 1, "cnt"], day.loc[day.workingday == 0, "cnt"],
                            label_a="working", label_b="non-working")
display(ht.format_result(res_tot, value_fmt="{:.0f}"))

Xd = pd.get_dummies(day[["workingday", "yr", "season", "weather", "temp", "hum", "windspeed"]],
                    columns=["season", "weather"], drop_first=True, dtype=float)
reg_tot = ht.ols_hac(day.cnt, Xd, maxlags=7)
display(reg_tot.round({"coef": 1, "hac_se": 1, "t": 2, "p_value": 4, "ci_low": 1, "ci_high": 1}))
print(f"R² = {reg_tot.attrs['r2']:.3f}")
pd.DataFrame([res_tot]).to_csv("outputs/model_results/hypothesis_daily_total.csv", index=False)

**The suggested test – daily totals:**
- Means **4,585 (working) vs 4,330 (non-working)**, difference **255/day, 95% CI [−58, 567]**, Welch t = 1.60, **p = 0.11**, Hedges' g = **0.13** (negligible). Mann–Whitney p = 0.12. → **Fail to reject H0.** (Levene p = 0.04 → variances differ, so Welch's version is the right one here.)
- With controls (same regression approach), working days have **+175 rentals/day (95% CI 27–324, p ≈ 0.02)** – about **4%** of average daily demand. Year (+2,009/day) and rain/snow (−1,833/day) matter far more.

**Combined message (one sentence for the interview):** *Day type barely changes **how many** rides happen (≈ 4%), but it massively changes **when** they happen (+24 percentage points in the peak windows) – so the availability problem is about timing and positioning, not daily volume.*

**Why both tests?** The suggested test alone gives a "no significant difference" result that could wrongly suggest working days don't matter. The peak-share test shows where the difference really is.

### 4.6 Design of experiment – testing the decision, not just the pattern
The tests above show *when* demand happens. The CEO's decision needs a **causal** answer: *does repositioning bikes fix availability more cheaply than buying bikes?* This dataset cannot answer that (no station data, no failed rentals), so I would run a pilot:

| Element | Design |
|---|---|
| **Operational hypothesis** | Pre-positioning bikes at high-demand stations before the commute peaks reduces empty-station time during peaks, without adding bikes. |
| **Treatment** | Rebalancing trucks fill the top morning-origin stations before 07:00 and the top evening-origin (office-area) stations before 16:00. |
| **Control** | Current operations. |
| **Randomisation** | *Switchback design*: working days are randomly assigned to treatment or control within each week (or station zones are randomised), so weather, season and growth are balanced between arms. |
| **Primary metric** | Minutes with zero bikes at target stations during 07–09 and 16–19 (from dock-status logs). |
| **Secondary metrics** | Peak-hour rentals, "no bike available" app events, complaints, rebalancing cost (truck-hours, km). |
| **Guardrails** | Empty/full-dock minutes at non-target stations (are we just moving the problem?). |
| **Duration and power** | 4–6 weeks; sample size set from the historical day-to-day variance of empty-station minutes. |
| **Decision rule** | If rebalancing brings peak availability to target at a lower cost per avoided stockout than the yearly cost of an extra bike → **don't buy**; buy only for stations that stay empty *after* rebalancing. |

## Phase 2 summary

**Completed**
- Cleaning applied and logged (humidity imputed, weather 4 merged, wind zeros and storm days flagged, nothing deleted).
- 21 model features, each with a documented reason, plus an explicit list of excluded columns (leakage, redundancy).
- 12 EDA charts with interpretations, plus the hypothesis charts (saved in `outputs/charts/`).
- Hypothesis formed on 2011 and confirmed on 2012: peak share is **+23.8 pp** higher on working days (p ≈ 10⁻¹¹⁹; holds after controls). The suggested daily-total test is **not significant** (p = 0.11; +4% after controls).
- Experiment design for the operational decision.

**Carried into Phase 3**
- Target `cnt`; features = `fe.MODEL_FEATURES`. For the linear model, drop `is_weekend` (collinear).
- Chronological split with training through June 2012; the test set (Jul–Dec 2012) is untouched so far.
- Test a log target and a "same hour last week" lag variant instead of assuming them.

**Limitations still open**
- No station data, trip durations, or failed rentals, so shortages can't be observed directly.
- Night averages are slightly overstated (zero-rental hours are missing from the file).
- 2011–12 growth makes any multi-year fleet projection uncertain.